# Driver Behaviour Detection - Google Colab
Este notebook permite entrenar y evaluar el modelo YOLOv11 para detección de comportamiento del conductor directamente en Google Colab con GPU gratuita.

**Clases:** Distracted, Drowsy, Eating, No seatbelt, Seatbelt, Smoking

## 1. Verificar GPU disponible

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU disponible: {torch.cuda.get_device_name(0)}')
    print(f'Memoria GPU: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('No hay GPU disponible. Ve a Runtime > Change runtime type > GPU')

## 2. Instalar dependencias

In [ ]:
!pip install ultralytics roboflow anvil-uplink ffmpeg-python -q

## 3. Descargar dataset desde Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="yxf6ArlOA5Yxvo5uE1Ty")
project = rf.workspace("driver-miviz").project("pta-s7fnu-nzeqr")
version = project.version(2)
dataset = version.download("yolov11")

print(f'Dataset descargado en: {dataset.location}')

## 4. Visualizar muestras del dataset

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

img_path = os.path.join(dataset.location, 'train', 'images')
label_path = os.path.join(dataset.location, 'train', 'labels')

classname = ['Distracted', 'Drowsy', 'Eating', 'No seatbelt', 'Seatbelt', 'Smoking']
num_data = 15

all_imgs = sorted(os.listdir(img_path))
all_labels = sorted(os.listdir(label_path))

selected_indices = random.sample(range(len(all_imgs)), num_data)
imgs = [all_imgs[i] for i in selected_indices]
labels = [all_labels[i] for i in selected_indices]

fig, axes = plt.subplots(3, 5, figsize=(21, 15))

for idx, ax in enumerate(axes.flat):
    if idx < len(imgs):
        try:
            img_path_full = os.path.join(img_path, imgs[idx])
            img = np.array(Image.open(img_path_full))
            img_h, img_w, _ = img.shape

            label_path_full = os.path.join(label_path, labels[idx])
            with open(label_path_full, 'r') as file:
                ann = file.read().strip().split()

                if len(ann) < 5:
                    print(f'Skipping {labels[idx]}: Insufficient values')
                    continue

                label = int(ann[0])
                x, y, w, h = map(float, ann[1:5])
                x_center, y_center = x * img_w, y * img_h
                box_w, box_h = w * img_w, h * img_h

                x1 = x_center - box_w // 2
                y1 = y_center - box_h // 2

                bbox = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor='red', facecolor='none')
                ax.add_patch(bbox)
                ax.set_title(f'{classname[label]}', fontsize=20)
            ax.imshow(img)
            ax.axis('off')
        except Exception as e:
            print(f'Error with {imgs[idx]}: {e}')
            ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Entrenar modelo YOLOv11

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt', verbose=False)

DATA_YAML = os.path.join(dataset.location, 'data.yaml')

model.train(
    data=DATA_YAML,
    epochs=50,
    patience=5,
    save=True,
    device=0,  # GPU 0 de Colab
    project='MOT'
)

## 6. Validar modelo entrenado

In [ ]:
# Cargar el mejor modelo entrenado
best_model = YOLO('MOT/train/weights/best.pt', verbose=True)

# Evaluar en el set de validación
results = best_model.val(data=DATA_YAML)

print(f"\nmAP50: {results.box.map50:.4f}")
print(f"mAP50-95: {results.box.map:.4f}")

## 7. Visualizar resultados del entrenamiento

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# Mostrar curvas de entrenamiento
results_img = glob.glob('MOT/train*/results.png')
if results_img:
    display(IPImage(filename=results_img[-1], width=900))

# Mostrar matriz de confusión
conf_img = glob.glob('MOT/train*/confusion_matrix.png')
if conf_img:
    display(IPImage(filename=conf_img[-1], width=700))

## 8. Probar predicciones en imágenes de test

In [ ]:
import random

test_images_path = os.path.join(dataset.location, 'test', 'images')
test_images = os.listdir(test_images_path)
sample_images = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, ax in enumerate(axes.flat):
    if idx < len(sample_images):
        img_file = os.path.join(test_images_path, sample_images[idx])
        results = best_model.predict(img_file, conf=0.25, verbose=False)
        annotated = results[0].plot()
        ax.imshow(annotated[:, :, ::-1])  # BGR a RGB
        ax.axis('off')

plt.suptitle('Predicciones en imágenes de test', fontsize=20)
plt.tight_layout()
plt.show()

## 9. Descargar pesos entrenados

In [ ]:
from google.colab import files

# Descargar el mejor modelo
files.download('MOT/train/weights/best.pt')
print('Pesos descargados exitosamente.')